# Lesson 5: Temperature, top-k, and top-p sampling

Explore how the same trained model produces different continuations when the sampling rule changes.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Load and tokenize text

We reuse `input.txt`, resolving it from the notebook folder or repository root. `B` means batch size, `T` means context length, and `C` means vector width. Our file is only 80 characters, so the validation scores are noisy and text generation will be limited.


In [ ]:
input_path = Path('input.txt')
if not input_path.is_file():
    input_path = Path('chatgpt/input.txt')
text = input_path.read_text(encoding='utf-8')
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.8 * len(data))
train_data, val_data = data[:split], data[split:]
block_size = min(8, len(train_data) - 1, len(val_data) - 1)
if block_size < 1:
    raise ValueError('input.txt needs enough text for train and validation sequences.')
batch_size = 4
print('Characters:', vocab_size, '| train:', len(train_data), '| validation:', len(val_data))
print('Context length:', block_size)


## Draw random batches

Choose starting positions within one split. Targets are the same window shifted right by one token. No window crosses from training into validation. The tokenizer vocabulary uses the full text so every validation character has an ID; model weights are updated only on training tokens.


In [ ]:
def get_batch(split_name='train'):
    if split_name not in ('train', 'val'):
        raise ValueError("Choose 'train' or 'val'.")
    source = train_data if split_name == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print('Input shape:', xb.shape, '| target shape:', yb.shape)
print('Input :', repr(decode(xb[0])))
print('Target:', repr(decode(yb[0])))
assert torch.equal(xb[:, 1:], yb[:, :-1])


## Causal multi-head attention

Each head compares queries to keys, then combines value vectors. A triangular mask prevents reading future tokens. This is the implementation developed in Lessons 3–4, included here so this notebook runs independently.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_size = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.projection = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('causal_mask', torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Give each head its own vector slice: [B, heads, T, head_size].
        q = q.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        k = k.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        v = v.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        weights = self.dropout(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, C)
        return self.projection(out)


## Transformer block

LayerNorm normalizes each token's features. Residual additions let information pass around attention and the MLP. The MLP expands each token vector, applies a nonlinear function, and projects it back.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, 4 * width), nn.GELU(),
            nn.Linear(4 * width, width), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


## Token and position embeddings → vocabulary scores

Token embeddings describe characters; learned position embeddings distinguish their positions. The final linear layer predicts the next character at every position. Cross entropy consumes raw scores (logits).


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, context_length, width=32, heads=4, layers=2):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context_length, width)
        self.blocks = nn.Sequential(*[
            TransformerBlock(width, heads, context_length) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.context_length:
            raise ValueError('Sequence exceeds context length.')
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.lm_head(self.final_norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

model = TinyGPT(vocab_size, block_size).to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## Train a small model

This cell continues training if rerun. Rerun the model-creation cell first for a fresh model. The small default run teaches the workflow; useful generation needs substantially more text and training.


In [ ]:
training_steps = 120
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()
for step in range(training_steps):
    x, y = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 40 == 0 or step == training_steps - 1:
        print(f'step {step}: loss {loss.item():.4f}')


## Sampling controls

- **Temperature:** divide logits by a positive number before softmax. Lower values concentrate probability; higher values spread it out. Our special value 0 chooses the highest score directly.
- **Top-k:** keep the k highest-scoring tokens.
- **Top-p:** keep the smallest ranked prefix whose cumulative probability reaches p.

These controls change which predictions are sampled; they do not improve the model's learned knowledge.


In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=80, temperature=1.0, top_k=None, top_p=None):
    if not prompt:
        raise ValueError('Provide a nonempty prompt using characters from input.txt.')
    if temperature < 0:
        raise ValueError('temperature must be nonnegative; 0 selects greedily.')
    if top_k is not None and top_k < 1:
        raise ValueError('top_k must be at least 1.')
    if top_p is not None and not 0 < top_p <= 1:
        raise ValueError('top_p must be in (0, 1].')
    target_device = next(model.parameters()).device
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=target_device)
    was_training = model.training
    model.eval()
    try:
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -model.context_length:])
            logits = logits[:, -1, :]
            if temperature == 0:
                next_id = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    threshold = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1:]
                    logits = logits.masked_fill(logits < threshold, float('-inf'))
                if top_p is not None:
                    sorted_logits, sorted_ids = logits.sort(descending=True)
                    cumulative = sorted_logits.softmax(dim=-1).cumsum(dim=-1)
                    # Keep the first token that takes cumulative mass over p.
                    remove = cumulative > top_p
                    remove[:, 1:] = remove[:, :-1].clone()
                    remove[:, 0] = False
                    sorted_logits = sorted_logits.masked_fill(remove, float('-inf'))
                    logits = torch.full_like(logits, float('-inf')).scatter(1, sorted_ids, sorted_logits)
                next_id = torch.multinomial(logits.softmax(dim=-1), num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return decode(idx[0].tolist())
    finally:
        model.train(was_training)


In [ ]:
examples = [
    ('greedy', dict(temperature=0)),
    ('cool', dict(temperature=0.5)),
    ('warm', dict(temperature=1.3)),
    ('top-k', dict(temperature=1.0, top_k=4)),
    ('top-p', dict(temperature=1.0, top_p=0.8)),
]
for label, settings in examples:
    torch.manual_seed(123)
    print(label, ':', repr(generate(model, text[:1], **settings)))


## Inspect the next-token distribution


In [ ]:
with torch.no_grad():
    logits, _ = model(torch.tensor([encode(text[:1])], device=device))
for temperature in (0.5, 1.0, 1.5):
    probabilities = (logits[0, -1] / temperature).softmax(dim=-1)
    values, ids = probabilities.topk(min(5, vocab_size))
    print(temperature, [(decode([i]), round(p.item(), 3)) for p, i in zip(values, ids)])


## Try it yourself

Use a fixed seed to compare `top_p=0.5` with `top_p=0.95`. Try `top_k=1`: why does it behave like greedy decoding? Try a longer prompt made only of characters in `input.txt`.
